# Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from torchtnt.utils.flops import FlopTensorDispatchMode
from tqdm import tqdm

from utils import num_params
from utils.postprocess import load_result_dataframe, last_epoch_only

W0326 10:43:00.317000 11251 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [3]:
sns.set(font_scale=1.25, style="whitegrid")

# Ignore known warnings that come when constructing subnets.
warnings.filterwarnings("ignore", message=".*The parameter 'pretrained' is deprecated.*")
warnings.filterwarnings("ignore", message=".*Arguments other than a weight enum.*")
warnings.filterwarnings("ignore", message=".*already erased node.*")

# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# print(f"Using device = {device}")

NUM_CORES = os.cpu_count()
if hasattr(os, "sched_getaffinity"):
    # This function is only available on certain platforms. When running with Slurm, it can tell us the true
    # number of cores we have access to.
    NUM_CORES = len(os.sched_getaffinity(0))
print (f"Using {NUM_CORES} cores.")

Using 8 cores.


# Scan Files

Find and load all result files.

In [7]:
project_name = "moe"
proj_dir = Path("../experiments") / project_name
full_df = load_result_dataframe(proj_dir, calc_sizes=False)

100%|██████████| 21/21 [00:00<00:00, 2211.69it/s]


Filter down to just the final performance: i.e., the last epoch number.

In [8]:
final_df = last_epoch_only(full_df)

Remap to more presentable names.

In [9]:
# First just reset the whole index so we can use them as columns.
final_df.drop(columns="Step", inplace=True)  # This column is duplicated, delete one of them.
final_df.reset_index(inplace=True)

column_name_remap = {
    "arch": "Architecture",
    "adapter": "Adapter",
    "seed": "Seed",
    "num_params": "# params",
    "num_trainable_params": "# trainable params",
    "num_flops": "# FLOPs",
    "Overall/Train Loss": "Train Loss",
    "Overall/Train Accuracy": "Train Accuracy",
    "Overall/Test Accuracy": "Test Accuracy",
}
final_df.rename(columns=column_name_remap, inplace=True)

# Set a new, clean index.
final_df.set_index(["Architecture", "Seed", "Epoch", "Step"], inplace=True)
final_df

/var/folders/hh/f20wzr451rd3m04552cf7n4r0000gp/T/ipykernel_11251/200306417.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.drop(columns="Step", inplace=True)  # This column is duplicated, delete one of them.
/var/folders/hh/f20wzr451rd3m04552cf7n4r0000gp/T/ipykernel_11251/200306417.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.rename(columns=column_name_remap, inplace=True)


dataset  Time/Data  \
Architecture                          Seed  Epoch Step                         
baseline-resnet-cifar-resisc          12345 10    320        cars   0.000070   
                                                  1960   cifar100   0.000034   
                                                  150         dtd   0.000073   
                                                  1130    fer2013   0.000036   
                                                  40      flowers   0.000065   
                                                  3010       fmow   0.000064   
                                                  2960    food101   0.000034   
                                                  2350      mnist   0.000035   
                                                  10240      pcam   0.000096   
                                                  740    resisc45   0.000035   
                                                  780      sun397   0.000038   
stitch-all-stages-resnet-cifar-resisc 12345 10    320        cars   0.000069   
                                                  1960   cifar100   0.000116   
                                                  150         dtd   0.000128   
                                                  1130    fer2013   0.000052   
                                                  40      flowers   0.000084   
                                                  2960    food101   0.000063   
                                                  2350      mnist   0.000067   
                                                  10240      pcam   0.000065   
                                                  740    resisc45   0.000092   
                                                  780      sun397   0.000046   

                                                               LR      Loss  \
Architecture                          Seed  Epoch Step                        
baseline-resnet-cifar-resisc          12345 10    320    0.000049  3.156945   
                                                  1960   0.000049  0.056158   
                                                  150    0.000049  1.612287   
                                                  1130   0.000049  1.089502   
                                                  40     0.000049  1.426029   
                                                  3010   0.000049  2.454371   
                                                  2960   0.000049  2.531636   
                                                  2350   0.000049  0.077128   
                                                  10240  0.000049  0.267105   
                                                  740    0.000049  0.049373   
                                                  780    0.000049  2.198368   
stitch-all-stages-resnet-cifar-resisc 12345 10    320    0.000005  2.794657   
                                                  1960   0.000005  0.672788   
                                                  150    0.000005  1.695499   
                                                  1130   0.000005  0.853150   
                                                  40     0.000005  2.185804   
                                                  2960   0.000005  1.185809   
                                                  2350   0.000005  0.007936   
                                                  10240  0.000005  0.188536   
                                                  740    0.000005  0.534915   
                                                  780    0.000005  1.721149   

                                                         Time/Img Per Sec  \
Architecture                          Seed  Epoch Step                      
baseline-resnet-cifar-resisc          12345 10    320         2264.566620   
                                                  1960        1924.169190   
                                                  150         2202.404406   
              

In [13]:
final_df[["dataset", "Train Accuracy", "Test Accuracy"]]

dataset  \
Architecture                          Seed  Epoch Step              
baseline-resnet-cifar-resisc          12345 10    320        cars   
                                                  1960   cifar100   
                                                  150         dtd   
                                                  1130    fer2013   
                                                  40      flowers   
                                                  3010       fmow   
                                                  2960    food101   
                                                  2350      mnist   
                                                  10240      pcam   
                                                  740    resisc45   
                                                  780      sun397   
stitch-all-stages-resnet-cifar-resisc 12345 10    320        cars   
                                                  1960   cifar100   
                                                  150         dtd   
                                                  1130    fer2013   
                                                  40      flowers   
                                                  2960    food101   
                                                  2350      mnist   
                                                  10240      pcam   
                                                  740    resisc45   
                                                  780      sun397   

                                                         Train Accuracy  \
Architecture                          Seed  Epoch Step                    
baseline-resnet-cifar-resisc          12345 10    320          0.367019   
                                                  1960         0.952260   
                                                  150          0.605053   
                                                  1130         0.551237   
                                                  40           0.750980   
                                                  3010         0.427027   
                                                  2960         0.426007   
                                                  2350         0.969467   
                                                  10240        0.881672   
                                                  740          0.976614   
                                                  780          0.497280   
stitch-all-stages-resnet-cifar-resisc 12345 10    320          0.345285   
                                                  1960         0.804800   
                                                  150          0.454255   
                                                  1130         0.641344   
                                                  40           0.448039   
                                                  2960         0.643208   
                                                  2350         0.993667   
                                                  10240        0.953060   
                                                  740          0.839577   
                                                  780          0.585743   

                                                         Test Accuracy  
Architecture                          Seed  Epoch Step                  
baseline-resnet-cifar-resisc          12345 10    320         0.177816  
                                                  1960        0.746300  
                                                  150         0.501596  
                                                  1130        0.550850  
                                                  40          0.542276  
                                                  3010        0.337208  
                                                  2960        0.508079  
                                                  2350        0.981700  
              

In [10]:
print("Available architectures:")
for aname in sorted(final_df.index.get_level_values("Architecture").unique()):
    print(f"    {aname}")

Available architectures:
    baseline-resnet-cifar-resisc
    stitch-all-stages-resnet-cifar-resisc


# Plots

In [18]:

def savefig(figname):
    # plt.tight_layout()  # Does not play nicely with Seaborn's FacetGrid. Using bbox_inches instead.
    save_dir = proj_dir / "figures"
    save_dir.mkdir(parents=True, exist_ok=True)
    print(f"Saving plot: {save_dir}/{figname}.(pdf/png)")
    plt.savefig(save_dir / (figname + ".pdf"), bbox_inches="tight")
    plt.savefig(save_dir / (figname + ".png"), bbox_inches="tight")


def plot_accuracy(final_acc_df, src_stage=None, dest_stage=None, arch_choice=None, arch_col="Architecture",
                  adapter_choice=None, adapter_col="Adapter", name=None, save=True, **kwargs):
    # Filter down to chosen gaps.
    if src_stage:
        final_acc_df = final_acc_df.loc[final_acc_df.index.get_level_values("Src Stage") == src_stage]
    if dest_stage:
        final_acc_df = final_acc_df.loc[final_acc_df.index.get_level_values("Dest Stage") == dest_stage]

    # Interpret the choice parameters.
    if not arch_choice:
        arch_choice = DFLT_ARCHS
    elif isinstance(arch_choice, str):
        arch_choice = [arch_choice]
    if not adapter_choice:
        adapter_choice = DFLT_ADAPTERS
    elif isinstance(adapter_choice, str):
        adapter_choice = [adapter_choice]

    # Either filter down to one choice value, or tell Seaborn to plot them as hue/style.
    title = []
    if name:
        title.append(name)
    if len(arch_choice) == 1:
        final_acc_df = final_acc_df.loc[final_acc_df.index.get_level_values(arch_col) == arch_choice[0]]
        title.append(arch_choice[0])
    elif arch_col in final_acc_df.index.names:
        kwargs.setdefault("row", arch_col)
        kwargs.setdefault("row_order", arch_choice)
    src_name = str(src_stage) if src_stage else "All"
    dest_name = str(dest_stage) if dest_stage else "All"
    title.append(f"Stage {src_name} → {dest_name}")
    if len(adapter_choice) == 1:
        final_acc_df = final_acc_df.loc[final_acc_df.index.get_level_values(adapter_col) == adapter_choice[0]]
        title.extend(["-", adapter_choice[0]])
    elif adapter_col in final_acc_df.index.names:
        kwargs.setdefault("hue", adapter_col)
        kwargs.setdefault("hue_order", adapter_choice)
    title = " ".join(title)

    # Automatically prefer these palettes for a certain number of hue categories.
    if "hue" in kwargs:
        if "hue_order" in kwargs:
            # If the set of hues is defined, use that.
            num_hues = len(kwargs["hue_order"])
        else:
            # Otherwise, count the number of unique values in the hue column.
            huecol = (final_acc_df.index.get_level_values(kwargs["hue"]) if kwargs["hue"] in final_acc_df.index.names
                      else final_acc_df[kwargs["hue"]])
            num_hues = len(huecol.unique())
        if num_hues > 20:
            kwargs.setdefault("palette", "magma")
        elif num_hues >= 10:  # FIXME: temporary "=", until I make manual color assignments
            kwargs.setdefault("palette", "tab20")

    # Try to intelligently select which kind based on the x-axis type.
    orig_x = None
    orig_y = orig_arch_acc.get(arch_choice[0]) if len(arch_choice) == 1 else None
    if "x" not in kwargs:
        kwargs["x"] = "Dest Stage"
    if kwargs["x"] == "Dest Stage" or kwargs["x"] == "num_downsamples":
        kwargs.setdefault("kind", "line")
        xvar = "Depth"
    elif "params" in kwargs["x"].lower():
        kwargs.setdefault("kind", "line")
        kwargs.setdefault("linestyle", ":")
        xvar = "Size"
        if len(arch_choice) == 1:
            orig_x = orig_arch_params.get(arch_choice[0])
    elif "flops" in kwargs["x"].lower():
        kwargs.setdefault("kind", "line")
        kwargs.setdefault("linestyle", ":")
        xvar = "FLOPs"
        if len(arch_choice) == 1:
            orig_x = orig_arch_flops.get(arch_choice[0])
    else:
        xvar = kwargs["x"]

    # Insert some other styling recommendations.
    if kwargs.get("kind") == "line":
        # Add markers to lines (if not already configured).
        kwargs.setdefault("marker", "o")
        kwargs.setdefault("markersize", 6)
        kwargs.setdefault("errorbar", "sd")
    kwargs.setdefault("facet_kws", {}).setdefault("legend_out", True)

    # Finally... plot!
    fg = sns.relplot(final_acc_df.reset_index(), y="Test Accuracy", **kwargs)
    if not kwargs.get("row") and not kwargs.get("col"):
        ax = fg.ax
        ax.set_title(title)
        # If we only have one axes, add some info about this particular model.
        if orig_y is not None:
            if orig_x is not None:
                # Plot a point representing the original model.
                ax.scatter(orig_x, orig_y, c="grey", s=100, marker="*", zorder=100)
            else:
                # Plot a horizontal line representing the original model.
                ax.axhline(orig_y, color="grey", linestyle="--")
        # And optionally save.
        if save:
            savefig(f"{title.replace(' ', '-').replace('→', 'to')}-acc-vs-{xvar}".lower())
        return ax
    else:
        return fg


def plot_accuracy_vs_depth(final_acc_df, **kwargs):
    return plot_accuracy(final_acc_df, x="Dest Stage", **kwargs)


def plot_accuracy_vs_size(final_acc_df, **kwargs):
    if "# params" not in final_acc_df.index.names and "# params" not in final_acc_df.columns:
        return None
    return plot_accuracy(final_acc_df, x="# params", **kwargs)


def plot_accuracy_vs_flops(final_acc_df, **kwargs):
    if "# FLOPs" not in final_acc_df.index.names and "# FLOPs" not in final_acc_df.columns:
        return None
    return plot_accuracy(final_acc_df, x="# FLOPs", **kwargs)


def main_plots(arch_choice, **kwargs):
    plot_accuracy_vs_depth(best_of_lrs, src_stage=current_src_stage, arch_choice=arch_choice, **kwargs)
    plot_accuracy_vs_size(best_of_lrs, src_stage=current_src_stage, arch_choice=arch_choice, **kwargs)
    plot_accuracy_vs_flops(best_of_lrs, src_stage=current_src_stage, arch_choice=arch_choice, **kwargs)


# Sandbox

In [68]:
final_df.loc[
    final_df.index.get_level_values("Architecture").str.fullmatch(f"ViT-S") &
    final_df.index.get_level_values("Adapter").str.fullmatch(f".*Block.*") #&
    # (final_df.index.get_level_values("first_deleted_block") == 5)
]

first_deleted_block  \
Architecture Adapter                   Src Stage Dest Stage Seed  Epoch Step  core_arch                        
ViT-S        No Downsample, BasicBlock 1         1          12345 10    25030 ViT-S                        1   
             Downsample, BasicBlock    1         1          12345 10    25030 ViT-S                        1   
             No Downsample, BasicBlock 1         2          12345 10    25030 ViT-S                        1   
             Downsample, BasicBlock    1         2          12345 10    25030 ViT-S                        1   
             No Downsample, BasicBlock 1         3          12345 10    25030 ViT-S                        1   
             Downsample, BasicBlock    1         3          12345 10    25030 ViT-S                        1   
             No Downsample, BasicBlock 1         4          12345 10    25030 ViT-S                        1   
             Downsample, BasicBlock    1         4          12345 10    25030 ViT-S                        1   
             No Downsample, BasicBlock 2         2          12345 10    25030 ViT-S                        3   
             Downsample, BasicBlock    2         2          12345 10    25030 ViT-S                        3   
             No Downsample, BasicBlock 2         3          12345 10    25030 ViT-S                        3   
             Downsample, BasicBlock    2         3          12345 10    25030 ViT-S                        3   
             No Downsample, BasicBlock 2         4          12345 10    25030 ViT-S                        3   
             Downsample, BasicBlock    2         4          12345 10    25030 ViT-S                        3   
             No Downsample, BasicBlock 3         3          12345 10    25030 ViT-S                        5   
             Downsample, BasicBlock    3         3          12345 10    25030 ViT-S                        5   
             No Downsample, BasicBlock 3         4          12345 10    25030 ViT-S                        5   
             Downsample, BasicBlock    3         4          12345 10    25030 ViT-S                        5   

                                                                                         last_deleted_block  \
Architecture Adapter                   Src Stage Dest Stage Seed  Epoch Step  core_arch                       
ViT-S        No Downsample, BasicBlock 1         1          12345 10    25030 ViT-S                       1   
             Downsample, BasicBlock    1         1          12345 10    25030 ViT-S                       1   
             No Downsample, BasicBlock 1         2          12345 10    25030 ViT-S                       2   
             Downsample, BasicBlock    1         2          12345 10    25030 ViT-S                       2   
             No Downsample, BasicBlock 1         3          12345 10    25030 ViT-S                       4   
             Downsample, BasicBlock    1         3          12345 10    25030 ViT-S                       4   
             No Downsample, BasicBlock 1         4          12345 10    25030 ViT-S                       6   
             Downsample, BasicBlock    1         4          12345 10    25030 ViT-S                       6   
             No Downsample, BasicBlock 2         2          12345 10    25030 ViT-S                       3   
             Downsample, BasicBlock    2         2          12345 10    25030 ViT-S                       3   
             No Downsample, BasicBlock 2         3          12345 10    25030 ViT-S                       4   
             Downsample, BasicBlock    2         3          12345 10    25030 ViT-S                       4   
             No Downsample, BasicBlock 2         4          12345 10    25030 ViT-S                       6   
             Downsample, BasicBlock    2         4          12345 10    25030 ViT-S                       6   
             No Downsample, BasicBlock 3         3          12345 10    

In [77]:
# Find missing results.
for cfgfile in tqdm(list(sorted(proj_dir.rglob("config.yml")))):
    result_files = list(cfgfile.parent.glob("result*.pkl"))
    if not result_files:
        print(f"Missing: {cfgfile.parent}")

100%|██████████| 1831/1831 [00:00<00:00, 53052.84it/s]

Missing: ../experiments/across-scales/mobilenet-v3/train-mobilenet-v3-gap-1,4-adapter-bottleneck_no_downsample
Missing: ../experiments/across-scales/mobilenet-v3/train-mobilenet-v3-gap-1,4-adapter-linear_no_downsample
Missing: ../experiments/across-scales/mobilenet-v3/train-mobilenet-v3-gap-1,4-adapter-linear_relu_no_downsample
Missing: ../experiments/across-scales/mobilenet-v3/train-mobilenet-v3-gap-1,6-adapter-bottleneck_no_downsample
Missing: ../experiments/across-scales/mobilenet-v3/train-mobilenet-v3-gap-1,6-adapter-finetune
Missing: ../experiments/across-scales/mobilenet-v3/train-mobilenet-v3-gap-1,6-adapter-linear_no_downsample
Missing: ../experiments/across-scales/mobilenet-v3/train-mobilenet-v3-gap-1,6-adapter-linear_relu_no_downsample
Missing: ../experiments/across-scales/mobilenet-v3/train-mobilenet-v3-gap-3,6-adapter-finetune
Missing: ../experiments/across-scales/mobilenet-v3-lr-2.0e-4/train-mobilenet-v3-lr-2.0e-4-gap-1,6-adapter-finetune
Missing: ../experiments/across-scal

In [85]:
from matplotlib import colormaps

colormaps["tab20"].colors[0]

(0.12156862745098039, 0.4666666666666667, 0.7058823529411765)